# Sprint 3 — Modèle Avancé · MLOps · IA Explicable
## Projet BigData — Prédiction du Diabète (BRFSS 2015)
**FISE A4 INFO — Nancy · Auteur : Enzo Fraioli · Date : 2026-04-15**

---
### Livrable 3 — Optimisation avancée, MLOps et IA Explicable

Ce notebook couvre les 6 livrables du Sprint 3 :

| # | Livrable |
|---|----------|
| 1 | Architecture enrichie du modèle |
| 2 | Choix du framework de deep learning |
| 3 | Gestion du déséquilibre des classes |
| 4 | Implémentation de principes MLOps |
| 5 | IA Explicable + Développement durable |
| 6 | Rapport final (`Reports/sprint3_report.md`) |


## Cellule 1 — Setup & Imports

In [1]:
import warnings
warnings.filterwarnings("ignore")
import json, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
import mlflow
import mlflow.keras
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score,
    recall_score, accuracy_score
)
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE
import shap
import lime
import lime.lime_tabular

ROOT       = Path(".")
PROC_DIR   = ROOT / "Data" / "Processed"
PROC_IMBAL = ROOT / "Data" / "Processed" / "imbalanced"
MODEL_DIR  = ROOT / "models"
REPORT_DIR = ROOT / "Reports"
TARGET     = "Diabetes_binary"
SEED       = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow : {tf.__version__}")
print(f"MLflow     : {mlflow.__version__}")
print(f"SHAP       : {shap.__version__}")


TensorFlow : 2.21.0
MLflow     : 3.11.1
SHAP       : 0.51.0


## Cellule 2 — Chargement des données

In [2]:
train = pd.read_csv(PROC_DIR / "train.csv")
val   = pd.read_csv(PROC_DIR / "val.csv")
test  = pd.read_csv(PROC_DIR / "test.csv")

X_train = train.drop(columns=[TARGET]).values.astype("float32")
y_train = train[TARGET].values.astype("float32")
X_val   = val.drop(columns=[TARGET]).values.astype("float32")
y_val   = val[TARGET].values.astype("float32")
X_test  = test.drop(columns=[TARGET]).values.astype("float32")
y_test  = test[TARGET].values.astype("float32")

feature_names = [c for c in train.columns if c != TARGET]
input_dim = X_train.shape[1]

print(f"Train   : {X_train.shape}  — positifs : {y_train.mean()*100:.1f}%")
print(f"Val     : {X_val.shape}  — positifs : {y_val.mean()*100:.1f}%")
print(f"Test    : {X_test.shape}  — positifs : {y_test.mean()*100:.1f}%")
print(f"Features: {input_dim}")


FileNotFoundError: [Errno 2] No such file or directory: 'Data/Processed/train.csv'

---
## § 1 — Architecture Enrichie du Modèle

### 1.1 Justification

L'architecture Sprint 2 (64→32, 2 couches) établissait un baseline.
Sprint 3 compare trois architectures :

| Architecture | Couches | Neurones | Particularité |
|---|---|---|---|
| **Baseline** | 2 | 64→32 | Référence Sprint 2 |
| **Deep** | 3 | 256→128→64 | Plus large, plus profond |
| **Residual** | 4 | 128→128→64 | Skip connections (ResNet-style) |

**Pourquoi les skip connections ?**
Les connexions résiduelles permettent au gradient de circuler directement,
évitant le vanishing gradient dans les réseaux profonds.
Pour des features corrélées (CardioRisk, HighBP...), elles permettent d'apprendre
des transformations complexes TOUT EN conservant les signaux directs des features clés.


In [ ]:
def compile_model(model, lr=1e-3):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=["accuracy", keras.metrics.AUC(name="auc"),
                 keras.metrics.Precision(name="precision"),
                 keras.metrics.Recall(name="recall")]
    )
    return model


def build_baseline(input_dim, l2=1e-4, lr=1e-3):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(l2)),
        layers.BatchNormalization(), layers.Dropout(0.3),
        layers.Dense(32, activation="relu", kernel_regularizer=regularizers.l2(l2)),
        layers.BatchNormalization(), layers.Dropout(0.2),
        layers.Dense(1, activation="sigmoid"),
    ], name="baseline_64_32")
    return compile_model(model, lr)


def build_deep(input_dim, l2=1e-4, lr=1e-3):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(l2)),
        layers.BatchNormalization(), layers.Dropout(0.4),
        layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(l2)),
        layers.BatchNormalization(), layers.Dropout(0.3),
        layers.Dense(64, activation="relu",  kernel_regularizer=regularizers.l2(l2)),
        layers.BatchNormalization(), layers.Dropout(0.2),
        layers.Dense(1, activation="sigmoid"),
    ], name="deep_256_128_64")
    return compile_model(model, lr)


def build_residual(input_dim, l2=1e-4, lr=1e-3):
    inp = layers.Input(shape=(input_dim,))
    x   = layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(l2))(inp)
    x   = layers.BatchNormalization()(x)
    # Bloc résiduel 1
    skip = x
    x = layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(l2))(x)
    x = layers.BatchNormalization()(x); x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(l2))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, skip]); x = layers.Activation("relu")(x)
    # Bloc résiduel 2
    skip2 = layers.Dense(64)(x)
    x = layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(l2))(x)
    x = layers.BatchNormalization()(x); x = layers.Dropout(0.2)(x)
    x = layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(l2))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, skip2]); x = layers.Activation("relu")(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model(inputs=inp, outputs=out, name="residual_128_64")
    return compile_model(model, lr)


for name, fn in [("Baseline", build_baseline), ("Deep", build_deep), ("Residual", build_residual)]:
    m = fn(input_dim)
    params = m.count_params()
    print(f"{name:<10} : {m.name:<22} | {params:>8,} paramètres | Couches: {len(m.layers)}")


### 1.2 Résultats de comparaison (via train_advanced.py)

In [ ]:
with open(REPORT_DIR / "sprint3_results.json") as f:
    arch_results = json.load(f)

df_arch = pd.DataFrame(arch_results).T.reset_index()
df_arch.columns = ["Architecture", "Accuracy", "AUC-ROC", "F1-score",
                   "Precision", "Recall", "Train time (s)", "Epochs", "CO2 (kg)"]
df_arch = df_arch.round(4)
print("=" * 72)
print("  COMPARAISON DES ARCHITECTURES — Dataset équilibré (50/50)")
print("=" * 72)
display(df_arch[["Architecture", "AUC-ROC", "F1-score", "Recall", "Precision",
                 "Accuracy", "Epochs", "Train time (s)"]])


### 1.3 Courbes d'entraînement

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
arch_names = ["baseline", "deep", "residual"]
titles     = ["Baseline (64-32)", "Deep (256-128-64)", "Residual (128-64)"]

for ax, arch, title in zip(axes, arch_names, titles):
    hist_path = MODEL_DIR / f"history_{arch}.json"
    if hist_path.exists():
        with open(hist_path) as f:
            h = json.load(f)
        epochs = range(1, len(h["loss"]) + 1)
        ax.plot(epochs, h["loss"],     label="Train loss",  color="#2196F3")
        ax.plot(epochs, h["val_loss"], label="Val loss",    color="#F44336", linestyle="--")
        ax.set_title(title, fontsize=11, fontweight="bold")
        ax.set_xlabel("Époque"); ax.set_ylabel("Loss")
        ax.legend(fontsize=8); ax.grid(alpha=0.3)
        ax.text(0.98, 0.95, f"N={len(h['loss'])} epochs",
                ha="right", va="top", transform=ax.transAxes, fontsize=8)

plt.suptitle("Courbes d'entraînement — Sprint 3", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(REPORT_DIR / "sprint3_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()


---
## § 2 — Choix du Framework de Deep Learning

### 2.1 Analyse comparative

| Critère | TensorFlow/Keras | PyTorch | XGBoost | LightGBM |
|---|---|---|---|---|
| Type | DL framework | DL framework | Gradient Boosting | Gradient Boosting |
| API haut niveau | Sequential API | Plus verbeux | sklearn-style | sklearn-style |
| Production/serving | TFServing | TorchServe | Direct | Direct |
| MLflow intégration | Native | Native | Native | Native |
| Données tabulaires | Bon | Bon | Excellent | Excellent |
| Flexibilité architectures | Haute | Haute | Limitée | Limitée |

### 2.2 Justification du choix TensorFlow/Keras

1. **API Sequential intuitive** : définition rapide, lisible
2. **Callbacks natifs** : EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
3. **mlflow.keras** : log automatique du modèle
4. **TFServing** : déploiement production REST/gRPC
5. **Functional API** : architectures résiduelles, multi-entrées
6. **Communauté** : documentation exhaustive, support long terme


In [ ]:
framework_results = {
    "Baseline NN": {"AUC-ROC": 0.8232, "F1": 0.7748, "Recall": 0.8456, "Precision": 0.7149, "Temps(s)": 7.4},
    "Deep NN (256-128-64)": {"AUC-ROC": 0.8229, "F1": 0.7723, "Recall": 0.8413, "Precision": 0.7137, "Temps(s)": 26.8},
    "Residual NN": {"AUC-ROC": 0.8186, "F1": 0.7705, "Recall": 0.8551, "Precision": 0.7010, "Temps(s)": 18.4},
    "XGBoost (300 trees)": {"AUC-ROC": 0.8221, "F1": 0.7707, "Recall": 0.8379, "Precision": 0.7131, "Temps(s)": 0.6},
    "LightGBM (300 trees)": {"AUC-ROC": 0.8233, "F1": 0.7723, "Recall": 0.8409, "Precision": 0.7127, "Temps(s)": 1.6},
}

df_fw = pd.DataFrame(framework_results).T.reset_index()
df_fw.columns = ["Modele", "AUC-ROC", "F1-score", "Recall", "Precision", "Temps (s)"]
print("=" * 75)
print("  COMPARAISON FRAMEWORKS — Dataset equilibre (50/50), seuil=0.45")
print("=" * 75)
display(df_fw)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
metrics = ["AUC-ROC", "F1-score", "Recall"]
colors  = ["#2196F3", "#2196F3", "#2196F3", "#FF9800", "#FF9800"]

for ax, metric in zip(axes, metrics):
    bars = ax.barh(df_fw["Modele"], df_fw[metric], color=colors)
    ax.set_xlabel(metric, fontsize=10)
    ax.set_title(f"Comparaison — {metric}", fontsize=11, fontweight="bold")
    ax.set_xlim(0.6, 0.95)
    for bar, val in zip(bars, df_fw[metric]):
        ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
                f"{val:.4f}", va="center", fontsize=8)
    ax.grid(axis="x", alpha=0.3)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor="#2196F3", label="Neural Network (TF/Keras)"),
                   Patch(facecolor="#FF9800", label="Gradient Boosting")]
fig.legend(handles=legend_elements, loc="lower center", ncol=2, fontsize=9)
plt.suptitle("Comparaison Frameworks — Prediction Diabete", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(REPORT_DIR / "sprint3_framework_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print()
print("Observation : LightGBM et NN atteignent des performances quasi-identiques.")
print("Avantage NN : flexibilite architecturale, representations apprises.")
print("Avantage LightGBM : entraînement 10x plus rapide, pas de GPU necessaire.")


---
## § 3 — Gestion du Déséquilibre des Classes

### 3.1 Le problème du déséquilibre

Le dataset équilibré 50/50 est artificiel (sous-échantillonnage). La réalité :

| Dataset | Lignes | % Diabétiques | Usage |
|---|---|---|---|
| 5050split (Sprint 1/2) | 70,692 | 50.0% | Benchmark |
| binary_full (Sprint 3) | 253,680 | 15.3% | Réel |

Sans correction sur données déséquilibrées : le modèle prédit toujours 0
→ Accuracy = 84.7% mais Recall = 0 (inutile en médecine !)

### 3.2 Stratégies

1. **Sans correction** — entraînement direct
2. **Class weights** — w_1 = n_0/n_1 ≈ 3.27 (pénaliser les FN sur la minorité)
3. **SMOTE** — synthétiser des exemples artificiels par interpolation k-NN


In [ ]:
train_imbal = pd.read_csv(PROC_IMBAL / "train.csv")
y_imbal = train_imbal[TARGET].values

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = pd.Series(y_imbal).value_counts()
axes[0].bar(["Non-diabetique (0)", "Diabetique (1)"],
            [counts[0.0], counts[1.0]],
            color=["#4CAF50", "#F44336"], edgecolor="white", width=0.5)
axes[0].set_title("Distribution — Dataset non-equilibre", fontsize=11)
axes[0].set_ylabel("Nombre d'echantillons")
for i, (val) in enumerate([counts[0.0], counts[1.0]]):
    axes[0].text(i, val + 500, f"{int(val):,}\n({val/len(y_imbal)*100:.1f}%)",
                 ha="center", fontsize=10)

pos_before = int(y_imbal.sum())
neg_before = len(y_imbal) - pos_before
axes[1].bar(["Neg avant", "Pos avant", "Neg apres SMOTE", "Pos apres SMOTE"],
            [neg_before, pos_before, neg_before, neg_before],
            color=["#4CAF50", "#F44336", "#4CAF50", "#FF9800"],
            edgecolor="white", width=0.6)
axes[1].set_title("Impact SMOTE sur la distribution", fontsize=11)
axes[1].set_ylabel("Nombre d'echantillons")
axes[1].tick_params(axis="x", labelsize=9)

plt.tight_layout()
plt.savefig(REPORT_DIR / "sprint3_class_imbalance.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
imbalance_results = {
    "Sans correction":  {"AUC-ROC": 0.8192, "F1": 0.3289, "Recall": 0.2339, "Precision": 0.5534},
    "Class weights":    {"AUC-ROC": 0.8179, "F1": 0.4450, "Recall": 0.8265, "Precision": 0.3045},
    "SMOTE":            {"AUC-ROC": 0.8031, "F1": 0.4535, "Recall": 0.6430, "Precision": 0.3502},
}

df_imbal = pd.DataFrame(imbalance_results).T.reset_index()
df_imbal.columns = ["Strategie", "AUC-ROC", "F1-score", "Recall", "Precision"]
print("=" * 65)
print("  GESTION DU DESEQUILIBRE — Dataset non-equilibre (~15% positifs)")
print("=" * 65)
display(df_imbal)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
palette = ["#9E9E9E", "#2196F3", "#FF9800"]
for ax, metric in zip(axes, ["AUC-ROC", "F1-score", "Recall", "Precision"]):
    bars = ax.bar(df_imbal["Strategie"], df_imbal[metric], color=palette, width=0.5)
    ax.set_title(metric, fontsize=10, fontweight="bold")
    ax.set_ylim(0, 1.0)
    ax.set_xticks(range(len(df_imbal)))
    ax.set_xticklabels(df_imbal["Strategie"], rotation=20, ha="right", fontsize=8)
    for bar, val in zip(bars, df_imbal[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.02,
                f"{val:.3f}", ha="center", fontsize=8, fontweight="bold")
    ax.grid(axis="y", alpha=0.3)

plt.suptitle("Impact des strategies de gestion du desequilibre", fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(REPORT_DIR / "sprint3_imbalance_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print()
print("Conclusion:")
print("  Sans correction : Recall=0.23 -> manque 77% des diabetiques!")
print("  Class weights   : Recall=0.83 -> detecte 83% des diabetiques (recommande)")
print("  SMOTE           : Recall=0.64, F1=0.45 -> compromis interessant")


---
## § 4 — Implémentation de Principes MLOps

### 4.1 Architecture MLOps

```
Donnees (DVC) --> Pretraitement --> Entrainement (MLflow) --> Registre --> Deploiement
     |                                      |
Versionnement                       Suivi experiences
```

### 4.2 DVC — Versionnement des données

Setup réalisé dans ce sprint :
```bash
dvc init                           # Initialise DVC dans le repo Git
dvc add Data/Raw/*.csv             # Track les 3 datasets (>200MB total)
git add Data/Raw/*.csv.dvc         # Commit les pointeurs DVC (quelques KB)
git commit -m "track raw data with DVC"
```

**dvc.yaml** — Pipeline reproductible :
```yaml
stages:
  preprocess_balanced:   python Src/preprocessing.py
  preprocess_imbalanced: python Src/preprocess_imbalanced.py
  train_advanced:        python Src/train_advanced.py --arch all
  explain:               python Src/explainability.py
```

Avantages :
- `dvc repro` relance uniquement les étapes invalidées
- `dvc pull` pour récupérer les données depuis un remote (S3/GCS)
- Hash MD5 de chaque fichier pour la reproductibilité


In [ ]:
import subprocess

# Vérifier l'état DVC
result = subprocess.run(["dvc", "status"], capture_output=True, text=True)
print("Etat DVC :")
print(result.stdout if result.stdout.strip() else "Pipeline a jour (aucun changement detecte)")

# Lister les fichiers .dvc
dvc_files = list(Path("Data/Raw").glob("*.dvc"))
print(f"\nFichiers DVC trackes ({len(dvc_files)}) :")
for f in dvc_files:
    size_kb = f.stat().st_size / 1024
    # Lire le hash depuis le .dvc file
    content = f.read_text()
    lines = [l for l in content.split("\n") if "md5" in l]
    md5 = lines[0].split(": ")[1].strip() if lines else "N/A"
    print(f"  {f.name:<65} ({size_kb:.1f} KB) | md5: {md5[:12]}...")


### 4.3 MLflow — Suivi expériences et registre de modèles

In [ ]:
mlflow.set_tracking_uri("sqlite:///mlruns/mlflow.db")
client = mlflow.tracking.MlflowClient()

experiments = client.search_experiments()
print("Experiences MLflow :")
for e in experiments:
    runs = client.search_runs(experiment_ids=[e.experiment_id])
    print(f"  [{e.experiment_id}] {e.name:<42} — {len(runs)} runs")

print()
for exp_name in ["diabetes_sprint3_balanced", "diabetes_sprint3_imbalanced"]:
    exp = client.get_experiment_by_name(exp_name)
    if exp is None:
        continue
    runs = client.search_runs(
        experiment_ids=[exp.experiment_id],
        order_by=["metrics.auc_roc DESC"],
        max_results=3
    )
    print(f"Top 3 — {exp_name}:")
    for r in runs:
        arch = r.data.params.get("architecture", "?")
        auc  = r.data.metrics.get("auc_roc", 0)
        f1   = r.data.metrics.get("f1", 0)
        print(f"  {r.info.run_id[:8]}... | {arch:<25} | AUC={auc:.4f} | F1={f1:.4f}")
    print()

# Model Registry
try:
    registered = client.search_registered_models()
    if registered:
        print("Registre de modeles MLflow :")
        for rm in registered:
            versions = client.search_model_versions(f"name='{rm.name}'")
            for v in versions:
                print(f"  {rm.name} v{v.version} — {v.current_stage} | Run={v.run_id[:8]}...")
except Exception as e:
    print(f"[info] {e}")

print()
print("Pour visualiser : python -m mlflow ui --backend-store-uri sqlite:///mlruns/mlflow.db")
print("                  -> http://localhost:5000")


---
## § 5 — IA Explicable et Développement Durable

### 5.1 Pourquoi l'explicabilité en santé ?

- **Confiance clinique** : le médecin doit comprendre la prédiction
- **Détection de biais** : vérifier que le modèle n'utilise pas des proxies problématiques
- **RGPD** : droit à l'explication des décisions algorithmiques
- **Amélioration itérative** : identifier les features sous-utilisées

### 5.2 SHAP — Analyse Globale

SHAP (SHapley Additive exPlanations) mesure la contribution marginale de chaque feature
en moyennant sur toutes les permutations possibles (théorie de Shapley).


In [ ]:
from IPython.display import Image, display as ipy_display
print("=== SHAP Summary Plot — Importance globale des features ===")
ipy_display(Image(filename=str(REPORT_DIR / "shap_summary.png"), width=650))


In [ ]:
print("=== SHAP Beeswarm — Distribution des impacts ===")
ipy_display(Image(filename=str(REPORT_DIR / "shap_beeswarm.png"), width=650))
print()
print("Lecture:")
print("  - Chaque point = un individu du test set")
print("  - Position X = valeur SHAP (impact sur la prediction)")
print("  - Couleur Rouge = feature avec valeur haute | Bleu = valeur basse")


### 5.3 SHAP — Explications locales (Waterfall)

In [ ]:
print("=== Cas POSITIF (diabetique predit) ===")
ipy_display(Image(filename=str(REPORT_DIR / "shap_waterfall_positive.png"), width=650))


In [ ]:
print("=== Cas NEGATIF (non-diabetique predit) ===")
ipy_display(Image(filename=str(REPORT_DIR / "shap_waterfall_negative.png"), width=650))


### 5.4 SHAP — Dependence Plots

In [ ]:
print("=== Dependence Plot — GenHlth (sante generale percue) ===")
ipy_display(Image(filename=str(REPORT_DIR / "shap_dependence_GenHlth.png"), width=600))
print()
print("=== Dependence Plot — BMI ===")
ipy_display(Image(filename=str(REPORT_DIR / "shap_dependence_BMI.png"), width=600))


### 5.5 LIME — Explications locales

In [ ]:
print("=== LIME — Cas POSITIF (diabetique predit) ===")
ipy_display(Image(filename=str(REPORT_DIR / "lime_positive.png"), width=650))


In [ ]:
print("=== LIME — Cas NEGATIF (non-diabetique predit) ===")
ipy_display(Image(filename=str(REPORT_DIR / "lime_negative.png"), width=650))
print()
print("LIME cree un modele lineaire local autour d'une prediction")
print("en perturbant les features et observant l'impact sur la sortie.")
print("Avantage sur SHAP : totalement agnostique au modele.")


### 5.6 Développement Durable — Empreinte Carbone (CodeCarbon)

In [ ]:
emissions_df = pd.read_csv(REPORT_DIR / "emissions.csv")
emissions_df["arch"] = emissions_df["project_name"].str.replace("diabetes_", "")

df_em = emissions_df[["arch", "duration", "emissions", "energy_consumed"]].copy()
df_em.columns = ["Architecture", "Duree (s)", "CO2 (kg)", "Energie (kWh)"]
df_em["CO2 (ug)"]    = (df_em["CO2 (kg)"] * 1e6).round(2)
df_em["Energie (mWh)"] = (df_em["Energie (kWh)"] * 1e6).round(2)

print("=" * 65)
print("  EMPREINTE CARBONE — CodeCarbon (France, Apple M4 Pro)")
print("=" * 65)
display(df_em[["Architecture", "Duree (s)", "CO2 (ug)", "Energie (mWh)"]].round(3))

total_co2_g = df_em["CO2 (kg)"].sum() * 1000
print(f"\nTotal CO2    : {total_co2_g:.4f} g CO2eq ({total_co2_g*1000:.2f} mg)")
print(f"Pays         : France (mix ~52g CO2/kWh)")
print(f"Equivalences :")
print(f"  = {total_co2_g/209*1000:.3f} m en voiture (209 g/km)")
print(f"  = {total_co2_g/0.185:.1f} s de streaming video HD")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

emissions_df_sorted = emissions_df.sort_values("emissions")
colors_bar = ["#4CAF50" if e < 1e-5 else "#FF9800" if e < 3e-5 else "#F44336"
              for e in emissions_df_sorted["emissions"]]
bars = axes[0].barh(emissions_df_sorted["arch"],
                    emissions_df_sorted["emissions"] * 1e6,
                    color=colors_bar, edgecolor="white")
axes[0].set_xlabel("CO2 equivalent (ug)", fontsize=10)
axes[0].set_title("Empreinte carbone par run", fontsize=11, fontweight="bold")
for bar, val in zip(bars, emissions_df_sorted["emissions"] * 1e6):
    axes[0].text(val + 0.3, bar.get_y() + bar.get_height()/2,
                 f"{val:.1f}ug", va="center", fontsize=8)
axes[0].grid(axis="x", alpha=0.3)

scatter_colors = ["#2196F3"]*3 + ["#FF9800"]*3
axes[1].scatter(emissions_df["duration"], emissions_df["emissions"]*1e6,
                c=scatter_colors, s=120, zorder=5, edgecolors="white", linewidths=1.5)
for _, row in emissions_df.iterrows():
    axes[1].annotate(row["arch"], (row["duration"], row["emissions"]*1e6),
                     textcoords="offset points", xytext=(5, 3), fontsize=7)
axes[1].set_xlabel("Duree entrainement (s)", fontsize=10)
axes[1].set_ylabel("CO2 (ug)", fontsize=10)
axes[1].set_title("Duree vs Empreinte carbone", fontsize=11, fontweight="bold")
axes[1].grid(alpha=0.3)

from matplotlib.patches import Patch
fig.legend(handles=[Patch(facecolor="#2196F3", label="Donnees equilibrees (50/50)"),
                    Patch(facecolor="#FF9800", label="Donnees non-equilibrees")],
           loc="lower center", ncol=2, fontsize=9)
plt.suptitle("Analyse de l'empreinte carbone — Sprint 3", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(REPORT_DIR / "sprint3_carbon_footprint.png", dpi=150, bbox_inches="tight")
plt.show()


---
## Conclusions — Sprint 3

### Bilan des livrables

| Livrable | Statut | Resultat principal |
|---|---|---|
| Architecture enrichie | OK | Baseline reste optimal en performance/CO2 |
| Framework DL | OK | TF/Keras justifie ; NN = LightGBM en AUC |
| Desequilibre classes | OK | Class weights : Recall 0.23→0.83 sur donnees reelles |
| MLOps (DVC + MLflow) | OK | Pipeline reproductible, registre de modeles |
| XAI (SHAP + LIME) | OK | GenHlth, HighBP, BMI = top 3 features |
| Developpement durable | OK | 0.134 g CO2eq total ; Baseline = meilleur compromis |

### Top 5 features explicatives (SHAP)

1. **GenHlth** — Sante generale percue (1-5, auto-evaluee)
2. **HighBP** — Hypertension arterielle diagnostiquee
3. **BMI** — Indice de masse corporelle
4. **Age** — Tranche d'age (1-13)
5. **CardioRisk** — Score composite risque cardiovasculaire

### Modele recommande en production

**Baseline NN** (64-32) avec **class_weight** pour la production reelle :
- AUC-ROC = 0.8232 (bon pouvoir discriminant)
- Recall = 0.8456 sur donnees equilibrees / 0.8265 sur donnees reelles
- CO2 = 5.7 µg (le plus economique)
- 4,801 parametres (leger, rapide en inference)

Rapport complet : `Reports/sprint3_report.md`
